# 15.5 Ranking metrics

Ranking metrics care not only what you retrieved, but how early the useful things appeared.

This metric-only notebook keeps the same F14 ladder but focuses on NDCG, MAP, and MRR. The important habit is matching the metric to the product surface and cutoff. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1500
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0.0:
        return 0.0
    return float(np.dot(a, b) / denom)


def ndcg_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    gains = (2.0 ** relevance[order] - 1.0) / np.log2(np.arange(2, len(order) + 2))
    ideal = np.argsort(-relevance, kind="mergesort")[:k]
    ideal_gains = (2.0 ** relevance[ideal] - 1.0) / np.log2(np.arange(2, len(ideal) + 2))
    denom = np.sum(ideal_gains)
    if denom == 0.0:
        return 0.0
    return float(np.sum(gains) / denom)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def recall_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float) > 0.0
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    denom = np.sum(relevance)
    if denom == 0:
        return 0.0
    return float(np.sum(relevance[order]) / denom)


def rating_ladder():
    d1 = np.array([
        [5.0, 4.0, 0.0, 1.0],
        [4.0, 5.0, 0.0, 1.0],
        [1.0, 1.0, 5.0, 4.0],
        [0.0, 1.0, 4.0, 5.0],
    ])
    d2 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0],
        [4.0, 5.0, 0.0, 1.0, 2.0],
        [1.0, 1.0, 5.0, 4.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0],
        [5.0, 0.0, 1.0, 0.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0],
    ])
    d3 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0, 2.0],
        [4.0, 4.0, 0.0, 0.0, 2.0, 0.0],
        [1.0, 1.0, 5.0, 4.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 3.0, 0.0, 3.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 1.0],
    ])
    d4 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0],
        [4.0, 5.0, 0.0, 2.0, 0.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [1.0, 0.0, 4.0, 5.0, 0.0, 5.0, 0.0, 2.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0],
        [0.0, 3.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 1.0, 4.0, 0.0],
        [0.0, 4.0, 0.0, 5.0, 0.0, 5.0, 0.0, 3.0],
        [4.0, 0.0, 2.0, 0.0, 5.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 4.0, 5.0, 0.0, 4.0, 0.0, 0.0],
    ])
    d5 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [4.0, 5.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 0.0, 5.0, 0.0, 0.0, 0.0, 2.0],
        [0.0, 0.0, 5.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 1.0, 0.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 5.0, 0.0, 3.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 4.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 5.0],
    ])
    return [
        {"name": "D1 tiny rating matrix", "ratings": d1, "holdout": [(0, 2, 5.0)]},
        {"name": "D2 synthetic preferences", "ratings": d2, "holdout": [(0, 2, 5.0), (2, 4, 2.0), (5, 4, 3.0)]},
        {"name": "D3 noise ties sparsity", "ratings": d3, "holdout": [(0, 2, 5.0), (1, 3, 2.0), (6, 1, 3.0), (7, 4, 2.0)]},
        {"name": "D4 inline MovieLens-like sample", "ratings": d4, "holdout": [(0, 2, 4.0), (1, 5, 3.0), (4, 3, 2.0), (6, 7, 2.0)]},
        {"name": "D5 long-tail cold-start", "ratings": d5, "holdout": [(0, 2, 4.0), (2, 9, 2.0), (5, 8, 4.0), (10, 0, 3.0), (11, 1, 3.0)]},
    ]


def slate_ladder():
    return [
        {"name": "D1 3-item slate", "features": np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.744, 0.643, 0.819]), "collab": np.array([0.30, 0.90, 0.70]), "relevance": np.array([2.0, 3.0, 1.0])},
        {"name": "D2 synthetic preferences", "features": np.array([[1.0, 0.1, 0.0], [0.2, 0.9, 0.8], [0.8, 0.7, 0.1], [0.1, 0.2, 1.0]]), "content": np.array([0.70, 0.62, 0.88, 0.55]), "collab": np.array([0.40, 0.85, 0.65, 0.30]), "relevance": np.array([1.0, 3.0, 2.0, 0.0])},
        {"name": "D3 noise ties sparsity", "features": np.array([[1.0, 0.0, 0.2], [0.9, 0.2, 0.0], [0.1, 1.0, 0.9], [0.2, 0.8, 0.8], [0.0, 0.1, 1.0]]), "content": np.array([0.78, 0.78, 0.59, 0.60, 0.51]), "collab": np.array([0.20, 0.50, 0.88, 0.40, 0.10]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 0.0])},
        {"name": "D4 inline MovieLens-like ratings", "features": np.array([[1.0, 0.3, 0.0], [0.9, 0.4, 0.1], [0.2, 0.9, 0.6], [0.1, 0.6, 1.0], [0.7, 0.2, 0.8], [0.0, 0.1, 0.9]]), "content": np.array([0.82, 0.80, 0.66, 0.59, 0.75, 0.50]), "collab": np.array([0.55, 0.61, 0.92, 0.72, 0.35, 0.25]), "relevance": np.array([2.0, 2.0, 3.0, 1.0, 2.0, 0.0])},
        {"name": "D5 long-tail cold-start", "features": np.array([[1.0, 0.2, 0.0], [0.8, 0.1, 0.1], [0.1, 0.9, 0.8], [0.0, 0.7, 1.0], [0.9, 0.9, 0.0], [0.1, 0.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.84, 0.78, 0.61, 0.58, 0.90, 0.52, 0.82]), "collab": np.array([0.65, 0.20, 0.91, 0.55, np.nan, 0.05, np.nan]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 3.0, 0.0, 2.0])},
    ]


def print_ladder_preview(rungs):
    for rung in rungs:
        if "ratings" in rung:
            matrix = rung["ratings"]
            observed = int(np.sum(matrix > 0.0))
            total = int(matrix.size)
            print(rung["name"], matrix.shape, "observed", observed, "of", total)
            print(matrix[:3, :5])
        else:
            relevance = rung["relevance"]
            print(rung["name"], "items", len(relevance), "positives", int(np.sum(relevance > 0.0)))
            print("relevance", relevance[:5])


## The concept, built once: NDCG, AP, and MRR

For graded relevance, $$DCG=\sum_irac{2^{rel_i}-1}{\log_2(i+2)},\qquad NDCG=rac{DCG}{IDCG}.$$ AP averages precision at every binary hit; MRR is the reciprocal rank of the first hit.

In [ ]:

def ranking_metrics(relevance, k=None):
    relevance = np.asarray(relevance, dtype=float)
    if k is None:
        k = len(relevance)
    rel_k = relevance[:k]
    discounts = np.log2(np.arange(2, len(rel_k) + 2))
    dcg = float(np.sum((2.0 ** rel_k - 1.0) / discounts))
    ideal = np.sort(relevance)[::-1][:k]
    idcg = float(np.sum((2.0 ** ideal - 1.0) / discounts))
    ndcg = 0.0 if idcg == 0.0 else dcg / idcg
    hits = (rel_k > 0.0).astype(float)
    precisions = []
    for idx, hit in enumerate(hits, start=1):
        if hit > 0.0:
            precisions.append(float(np.sum(hits[:idx]) / idx))
    ap = 0.0 if len(precisions) == 0 else float(np.mean(precisions))
    first_hits = np.where(hits > 0.0)[0]
    mrr = 0.0 if len(first_hits) == 0 else float(1.0 / (first_hits[0] + 1))
    return {"dcg": dcg, "idcg": idcg, "ndcg": ndcg, "ap": ap, "mrr": mrr}


## Check the lesson numbers

Graded relevance $(3,2,0,1)$ has $DCG=9.323$ and $NDCG=0.993$ because the ideal is $9.393$. Binary hits $(1,0,1,1,0)$ have $AP=0.806$, and hits $(0,0,1,1,0)$ have $MRR=0.333$.

In [ ]:

graded = ranking_metrics(np.array([3.0, 2.0, 0.0, 1.0]))
ap_case = ranking_metrics(np.array([1.0, 0.0, 1.0, 1.0, 0.0]))
mrr_case = ranking_metrics(np.array([0.0, 0.0, 1.0, 1.0, 0.0]))

assert np.round(graded["dcg"], 3) == 9.323
assert np.round(graded["ndcg"], 3) == 0.993
assert np.round(ap_case["ap"], 3) == 0.806
assert np.round(mrr_case["mrr"], 3) == 0.333

print("DCG", round(graded["dcg"], 3))
print("NDCG", round(graded["ndcg"], 3))
print("AP", round(ap_case["ap"], 3))
print("MRR", round(mrr_case["mrr"], 3))


## The dataset ladder

Each rung provides a relevance list after ranking. D3 includes ties and sparsity; D5 includes unjudged long-tail items that should not be silently confused with known bad items.

In [ ]:

rungs = slate_ladder()
print_ladder_preview(rungs)


In [ ]:

rows = []
for rung in rungs:
    scores = hybrid_score(rung["content"], rung["collab"], alpha=0.4, fallback_to_content=True)
    order = np.argsort(-scores, kind="mergesort")
    ranked_relevance = rung["relevance"][order]
    metrics = ranking_metrics(ranked_relevance, k=3)
    rows.append({"name": rung["name"], "ranked_relevance": ranked_relevance, "metric": metrics["ndcg"], "metrics": metrics})

for row in rows:
    print(f"{row['name']}: NDCG@3={row['metric']:.3f}, AP@3={row['metrics']['ap']:.3f}, MRR@3={row['metrics']['mrr']:.3f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
last = rows[-1]
axes[0].bar(np.arange(len(last["ranked_relevance"])), last["ranked_relevance"], color="seagreen")
axes[0].set_title("D5 relevance at rank")
axes[0].set_xlabel("rank")
axes[0].set_ylabel("relevance")
axes[1].plot([row["name"].split()[0] for row in rows], [row["metric"] for row in rows], marker="o")
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title("NDCG@3 vs sparsity rung")
axes[1].set_ylabel("NDCG@3")
fig.tight_layout()


## Pitfall on D5: inconsistent cutoffs and unjudged items

MRR can look good after the first hit while later relevant items are misplaced. Treating unjudged long-tail items as irrelevant also makes missing labels look like negative labels.

In [ ]:

d5 = slate_ladder()[-1]
scores = hybrid_score(d5["content"], d5["collab"], alpha=0.4, fallback_to_content=True)
order = np.argsort(-scores, kind="mergesort")
ranked = d5["relevance"][order]
wrong_mrr = ranking_metrics((ranked > 0.0).astype(float), k=3)["mrr"]
fixed_ndcg = ranking_metrics(ranked, k=3)["ndcg"]
longer_ndcg = ranking_metrics(ranked, k=min(6, len(ranked)))["ndcg"]

print("MRR@3 only", round(wrong_mrr, 3))
print("consistent graded NDCG@3", round(fixed_ndcg, 3))
print("different cutoff NDCG@6", round(longer_ndcg, 3))


## Evaluate it + Practice

Evaluation checklist:
- Track `NDCG@3` on D1--D5 and compare with a no-skill baseline such as popularity or original order.
- Run a sanity check where the strongest observed signal is swapped and the top item changes.
- Ablate the key idea, such as masking, latent factors, calibration, or query-local losses, and confirm the metric drops.
- Watch failure signals: identical recommendations for everyone, head-item dominance, unstable tiny-overlap scores, and poor cold-start behavior.

Practice:
1. Change one D3 tie and predict which slate position moves before running the cell.


2. Change one D5 unjudged item to unknown and exclude it from IDCG.
3. Compare MAP and NDCG when the top item is relevant but low grade.